In [8]:
import json

input_path = r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-4B-new\train.jsonl"
output_path = r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-4B-new\train_cleaned.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        if not line.strip():
            continue

        data = json.loads(line)

        # Remove leakage_spans if it is empty
        if data.get("leakage_spans") == []:
            continue

        fout.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"Saved cleaned file to: {output_path}")

Saved cleaned file to: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-4B-new\train_cleaned.jsonl


In [1]:
from pathlib import Path
import json
import random
import shutil


src = Path(
    r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-8B-new\train_cleaned.jsonl"
)

backup = src.with_suffix(src.suffix + ".bak")
test_path = src.with_name("test_" + src.name)
val_path = src.with_name("val_" + src.name)


# Create backup if it does not already exist
if not backup.exists():
    shutil.copy2(src, backup)


# Always split from the original backup so rerunning is safe
records = [
    json.loads(line)
    for line in backup.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Deterministic shuffle
random.Random(42).shuffle(records)


# Calculate split sizes
n = len(records)
n_test = round(n * 0.15)
n_val = round(n * 0.15)


# Create splits: 15% test, 15% validation, 70
# % train
splits = {
    test_path: records[:n_test],
    val_path: records[n_test:n_test + n_val],
    src: records[n_test + n_val:],
}


# Write each split
for path, rows in splits.items():
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# Print summary
print(f"total={n}")
print(f"test={len(splits[test_path])}: {test_path}")
print(f"val={len(splits[val_path])}: {val_path}")
print(f"train={len(splits[src])}: {src}")
print(f"backup={backup}")

total=1496
test=224: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-8B-new\test_train_cleaned.jsonl
val=224: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-8B-new\val_train_cleaned.jsonl
train=1048: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-8B-new\train_cleaned.jsonl
backup=H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-8B-new\train_cleaned.jsonl.bak


In [2]:
def harmonic_mean(a, b):
    """Calculate the harmonic mean of two numbers."""
    if a <= 0 or b <= 0:
        raise ValueError("Both numbers must be positive.")
    return 2 * (a * b) / (a + b)

In [3]:
harmonic_mean(0.9, 0.1)

0.18000000000000002

In [4]:
harmonic_mean(0.5, 0.5)

0.5

In [6]:
harmonic_mean(0.8, 0.4)

0.5333333333333333

In [ ]:
import os, re, json, asyncio
from pathlib import Path

from datasets import load_dataset
from openai import AsyncOpenAI
from tqdm.auto import tqdm


# =====================
# Config
# =====================
MODEL = "Qwen/Qwen3-4B"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"

N_SAMPLES = 100
MAX_CONCURRENCY = 8
MAX_NEW_TOKENS_NATIVE = 2048
MAX_NEW_TOKENS_REWRITE = 1024

OUT_DIR = Path("method/calibrate_leakage_detector/data/reasoning_trace_compare")
NATIVE_PATH = OUT_DIR / "native_reasoning_traces.jsonl"
NONNATIVE_PATH = OUT_DIR / "nonnative_reasoning_traces.jsonl"

OUT_DIR.mkdir(parents=True, exist_ok=True)

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)


# =====================
# Helpers
# =====================
def extract_think_and_answer(text):
    m = re.search(r"<think>(.*?)</think>", text, flags=re.S)
    if not m:
        return "", text.strip()

    native_think = m.group(1).strip()
    final_answer = text[m.end():].strip()
    return native_think, final_answer


async def complete(messages, max_tokens, enable_thinking=None):
    kwargs = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }

    # For Qwen-style vLLM servers. Remove this block if your endpoint rejects it.
    if enable_thinking is not None:
        kwargs["extra_body"] = {
            "chat_template_kwargs": {"enable_thinking": enable_thinking}
        }

    r = await client.chat.completions.create(**kwargs)
    return r.choices[0].message.content or ""


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# =====================
# Prompts
# =====================
def native_prompt(problem):
    return [
        {
            "role": "user",
            "content": f"Solve the following math problem.\n\nProblem:\n{problem}",
        }
    ]


def rewrite_prompt(problem, final_answer):
    return [
        {
            "role": "user",
            "content": (
                "Rewrite the reasoning for the following math problem as a clean, concise "
                "step-by-step solution. Do not mention that you are rewriting. Do not use "
                "<think> tags.\n\n"
                f"Problem:\n{problem}\n\n"
                f"Final answer to explain:\n{final_answer}"
            ),
        }
    ]


# =====================
# Main
# =====================
async def process_one(i, row, sem):
    problem = row["question"]
    gold_answer = row["answer"]

    async with sem:
        native_response = await complete(
            native_prompt(problem),
            max_tokens=MAX_NEW_TOKENS_NATIVE,
            enable_thinking=True,
        )

    native_trace, final_response = extract_think_and_answer(native_response)

    async with sem:
        rewritten_trace = await complete(
            rewrite_prompt(problem, final_response),
            max_tokens=MAX_NEW_TOKENS_REWRITE,
            enable_thinking=False,
        )

    native_row = {
        "sample_id": i,
        "problem": problem,
        "gold_answer": gold_answer,
        "native_think": native_trace,
        "final_response": final_response,
        "raw_response": native_response,
    }

    nonnative_row = {
        "sample_id": i,
        "problem": problem,
        "gold_answer": gold_answer,
        "nonnative_think": rewritten_trace.strip(),
        "source_final_response": final_response,
    }

    return native_row, nonnative_row


async def main():
    data = load_dataset("gsm8k", "main", split=f"test[:{N_SAMPLES}]")
    sem = asyncio.Semaphore(MAX_CONCURRENCY)

    tasks = [process_one(i, row, sem) for i, row in enumerate(data, 1)]

    native_rows, nonnative_rows = [], []
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        native_row, nonnative_row = await coro
        native_rows.append(native_row)
        nonnative_rows.append(nonnative_row)

    native_rows.sort(key=lambda x: x["sample_id"])
    nonnative_rows.sort(key=lambda x: x["sample_id"])

    write_jsonl(NATIVE_PATH, native_rows)
    write_jsonl(NONNATIVE_PATH, nonnative_rows)

    print(f"Saved native traces to: {NATIVE_PATH}")
    print(f"Saved nonnative traces to: {NONNATIVE_PATH}")


await main()

In [1]:
from datasets import load_dataset

ds = load_dataset("json", data_files=r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\Qwen3-4B-new\qwen3-4b-generated-response.jsonl")

h:\miniconda3\envs\LGM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 5000 examples [00:00, 22842.21 examples/s]


In [4]:
ds['train'][0]

{'generation_model': 'Qwen3-4B',
 'prompt': '# Carlos Andrés Mina\nCarlos Andrés Mina( born 10 October 1992) is an Ecuadorian amateur boxer.He competed in the light heavyweight division at the 2016 Summer Olympics, but was eliminated in the third bout.Mina is a composer and singer of rap music.In 2014 he recorded an album" La Tinta" under an alias Jeanthes Space.He starred in the documentary" La Tola Box" about his boxing gym La Tola and composed the soundtrack for it.His brother Nixon is an international basketball player, and his cousin Abel is a boxer.\n# Luis Piñerúa Ordaz\nLuis María Piñerúa Ordaz( 20 April 1924- 8 February 2001) was the Democratic Action presidential candidate in the 1978 Venezuelan general election, losing to COPEI\'s Luis Herrera Campins.He held the post of Minister of Home Affairs of Venezuela in both the First Presidency of Carlos Andrés Pérez and Second Presidency of Carlos Andrés Pérez.\n# Carlos Andrés Rivas\nCarlos Andrés Rivas Gómez( born 22 August 1991)

In [8]:
def cosine_similarity(a, b):
    """Calculate the cosine similarity between two 1D tensors."""
    if a.dim() != 1 or b.dim() != 1:
        raise ValueError("Both inputs must be 1D tensors.")
    if a.size(0) != b.size(0):
        raise ValueError("Both tensors must have the same length.")

    dot_product = torch.dot(a, b)
    norm_a = torch.norm(a)
    norm_b = torch.norm(b)

    if norm_a == 0 or norm_b == 0:
        raise ValueError("One of the tensors is zero, cannot compute cosine similarity.")

    return dot_product / (norm_a * norm_b)

In [15]:
import torch

a = torch.tensor([0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], dtype=torch.float32)
b = torch.tensor([0.2, 0.1, 0.05, 0.1, 1, 1, 0.2, 0.1, 0.2, 0.1, 0.2, 0.1, 0.2, 0.1, 0.2, 1, 1, 1, 0.1, 0.2, 0.1, 0.2, 0.1, 0.2, 0.1, 1], dtype=torch.float32)
c = torch.tensor([0.5] * 26, dtype=torch.float32)
d = b / 10

In [16]:
print(cosine_similarity(a, b))
print(cosine_similarity(a, c))
print(cosine_similarity(a, d))

tensor(0.8796)
tensor(0.4385)
tensor(0.8796)


In [ ]:
export OPENAI_API_KEY="...."

python method/calibrate_leakage_detector/create_test_detector.py \
  --input-path method/calibrate_leakage_detector/data/Qwen3-4B/test_generated-responses_Qwen3-4B_with-leakage-spans.jsonl \
  --output-path method/calibrate_leakage_detector/data/Qwen3-4B/test_counterfactual.jsonl \
  --model Qwen/Qwen3-4B \
  --generation-model Qwen3-4B \
  --generation-base-url http://localhost:8001/v1 \
  --judge-model gpt-5.6-luna \
  --judge-reasoning-effort xhigh \
  --judge-retries 5 \
  --judge-max-tokens 8192 \
  --base-url https://api.openai.com/v1 \
  --api-key .... \
  --device-map auto \
  --dtype bfloat16 \
  --max-concurrency 8 \
  --max-new-tokens 2048 \
  --max-repair-stages 5 \
  --generation-backend vllm \
  --overwrite

python method/calibrate_leakage_detector/create_test_detector.py \
  --input-path method/calibrate_leakage_detector/data/Qwen3-4B/test_generated-responses_Qwen3-4B_with-leakage-spans.jsonl \
  --output-path method/calibrate_leakage_detector/data/Qwen3-4B/test_counterfactual.jsonl \
  --model Qwen/Qwen3-4B \
  --generation-backend vllm \
  --generation-model Qwen3-4B \
  --generation-base-url http://localhost:8000/v1 \
  --judge-model gpt-5.6-luna \
  --judge-reasoning-effort xhigh \
  --base-url https://api.openai.com/v1 \
  --api-key .... \
  --max-concurrency 16 \
  --max-new-tokens 2048 \
  --max-repair-stages 5 \
  --overwrite


vllm serve Qwen/Qwen3-4B \
  --tensor-parallel-size 1 \
  --port 8000 \
  --gpu_memory_utilization 0.8 \
  --served-model-name Qwen3-4B